In [16]:
# Cellule 1 — Imports + config

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from env.workshop_env import WorkshopEnv

from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker

# Reproductibilité globale (pour la génération des seeds du protocole)
GLOBAL_SEED = 20251217
rng = np.random.default_rng(GLOBAL_SEED)

# Horizon : 1 semaine
WEEK_STEPS = 10080

print("OK imports. WEEK_STEPS =", WEEK_STEPS)


OK imports. WEEK_STEPS = 10080


In [18]:
# Cellule 2 — Charger / définir l’expert v4

import numpy as np

def expert_policy(obs: np.ndarray, env: WorkshopEnv) -> int:
    

    time = float(obs[0]) * float(env.max_time)

    m1_busy = int(round(obs[1]))
    m1_time_left = float(obs[2]) * 100.0

    m2_busy = int(round(obs[3]))
    m2_time_left = float(obs[4]) * 100.0

    stock_raw = float(obs[5]) * float(env.raw_capacity)
    stock_p1 = float(obs[6]) * float(env.raw_capacity)
    stock_p2_inter = float(obs[7]) * float(env.raw_capacity)
    stock_p2 = float(obs[8]) * float(env.raw_capacity)

    next_delivery_cd = float(obs[9]) * 10080.0

    demande_p1 = float(obs[10]) * 1000.0
    demande_p2 = float(obs[11]) * 1000.0
    q_raw_incoming = float(obs[12]) * 1000.0

    m1_free = (m1_busy == 0)
    m2_free = (m2_busy == 0)

    backlog_p1 = max(demande_p1, 0.0)
    backlog_p2 = max(demande_p2, 0.0)
    backlog_total = backlog_p1 + backlog_p2

    def choose_k(backlog, k_max=5):
        k_max_int = int(k_max)
        if k_max_int < 1:
            return 0
        if backlog <= 5:       k = 1
        elif backlog <= 15:    k = 2
        elif backlog <= 30:    k = 3
        elif backlog <= 60:    k = 4
        else:                  k = 5
        k = min(k, k_max_int)
        return max(1, int(k))

    target_raw = min(env.raw_capacity, backlog_total + 10.0)
    current_pipeline = max(0.0, stock_raw + q_raw_incoming)
    missing = target_raw - current_pipeline

    if missing > 0:
        k_cmd = int(max(1, min(50, missing)))
        action_cmd = 149 + k_cmd  # 150 → 199
        if 0 <= action_cmd <= 200:
            return action_cmd

    SEUIL_MIN_STOCK = 5.0
    LOT_PREFAB = 2

    if backlog_total <= 0.0:
        stock_produit = stock_p1 + stock_p2
        if stock_produit < SEUIL_MIN_STOCK and stock_raw > 0:
            if m1_free:
                k_prefab = min(LOT_PREFAB, int(min(5, stock_raw)))
                if k_prefab >= 1:
                    action_p2_step1 = 49 + k_prefab   # 50 → 99
                    if 0 <= action_p2_step1 <= 200:
                        return action_p2_step1
            if m1_free:
                k_prefab = min(LOT_PREFAB, int(min(5, stock_raw)))
                if k_prefab >= 1:
                    action_p1 = k_prefab - 1          # 0 → 49
                    if 0 <= action_p1 <= 200:
                        return action_p1

    if m2_free and stock_p2_inter > 0:
        k2 = choose_k(backlog_p2, k_max=min(5, stock_p2_inter))
        if k2 > 0:
            action_p2_step2 = 99 + k2  # 100 → 149
            if 0 <= action_p2_step2 <= 200:
                return action_p2_step2

    if m1_free and backlog_p2 > 0 and stock_raw > 0:
        k1_p2 = choose_k(backlog_p2, k_max=min(5, stock_raw))
        if k1_p2 > 0:
            action_p2_step1 = 49 + k1_p2  # 50 → 99
            if 0 <= action_p2_step1 <= 200:
                return action_p2_step1

    if m1_free and backlog_p1 > 0 and stock_raw > 0:
        k1_p1 = choose_k(backlog_p1, k_max=min(5, stock_raw))
        if k1_p1 > 0:
            action_p1 = k1_p1 - 1         # 0 → 49
            if 0 <= action_p1 <= 200:
                return action_p1

    return 200


def expert_policy_masked(env: WorkshopEnv, obs: np.ndarray) -> int:
    
    mask = env.get_action_mask().astype(bool)
    a = expert_policy(obs, env)

    if not isinstance(a, (int, np.integer)):
        a = 200
    elif a < 0 or a >= len(mask):
        a = 200

    if not mask[a]:
        if mask[200]:
            return 200
        valid_actions = np.where(mask)[0]
        return int(valid_actions[0])

    return int(a)

print("Expert v4 policy loaded.")


Expert v4 policy loaded.


In [20]:
# Cellule 3 — Charger PPO v7 (best_model)

def mask_fn(env: WorkshopEnv):
    return env.get_action_mask()

# Env "dummy" pour charger le modèle (MaskablePPO veut un env avec masker)
env_load = ActionMasker(WorkshopEnv(), mask_fn)

PPO_V7_PATH = "./ppo_safe_best_v7/best_model.zip"  
assert os.path.exists(PPO_V7_PATH), f"Introuvable: {PPO_V7_PATH}"

ppo_v7 = MaskablePPO.load(PPO_V7_PATH, env=env_load, device="cuda")  
print("PPO v7 loaded:", PPO_V7_PATH)


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
PPO v7 loaded: ./ppo_safe_best_v7/best_model.zip


In [ ]:
# Cellule — Courbes de reward cumulé sur UNE semaine (Expert v4 vs PPO v7)

# -------- paramètres --------
SEED_TO_PLOT = 50_000     
WEEK_STEPS = 10_080       # 1 semaine
N_POINTS = 100
# ----------------------------

def run_week_with_trace_expert(seed, max_steps=WEEK_STEPS):
    env = WorkshopEnv()
    obs, _ = env.reset(seed=seed)
    rewards = []

    for _ in range(max_steps):
        a = expert_policy_masked(env, obs)
        obs, r, terminated, truncated, _ = env.step(a)
        rewards.append(r)
        if terminated or truncated:
            break

    return np.array(rewards)


def run_week_with_trace_ppo(seed, model, max_steps=WEEK_STEPS):
    env = WorkshopEnv()
    obs, _ = env.reset(seed=seed)
    rewards = []

    for _ in range(max_steps):
        mask = env.get_action_mask()
        a, _ = model.predict(obs, deterministic=True, action_masks=mask)
        obs, r, terminated, truncated, _ = env.step(a)
        rewards.append(r)
        if terminated or truncated:
            break

    return np.array(rewards)


# --- Exécution de la semaine ---
r_exp = run_week_with_trace_expert(SEED_TO_PLOT)
r_ppo = run_week_with_trace_ppo(SEED_TO_PLOT, ppo_v7)

# Rewards cumulés
cum_exp = np.cumsum(r_exp)
cum_ppo = np.cumsum(r_ppo)

# Sous-échantillonnage (~100 points)
idx = np.linspace(0, len(cum_exp) - 1, N_POINTS).astype(int)

# Plot
plt.figure(figsize=(8, 4))
plt.plot(idx, cum_exp[idx], label="Expert v4", linewidth=2)
plt.plot(idx, cum_ppo[idx], label="PPO v7", linewidth=2)

plt.xlabel("Time within the week (subsampled)")
plt.ylabel("Cumulative reward")
plt.title(f"Cumulative reward over one week (seed = {SEED_TO_PLOT})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [9]:
# Cellule 4 — Définir les semaines de test 
#(104 seeds “hors entraînement”)

N_WEEKS = 104

# Choix simple, lisible, éloigné : seeds 50000..50103
TEST_BASE_SEED = 50_000
test_seeds = list(range(TEST_BASE_SEED, TEST_BASE_SEED + N_WEEKS))

print("N_WEEKS =", N_WEEKS)
print("Seed range:", test_seeds[0], "→", test_seeds[-1])


N_WEEKS = 104
Seed range: 50000 → 50103


In [11]:
# Cellule 5 — Runner “une semaine” pour expert et PPO 
# (même protocole)

def run_week_expert(seed: int, max_steps=WEEK_STEPS) -> float:
    env = WorkshopEnv()
    obs, _ = env.reset(seed=seed)
    total = 0.0

    for _ in range(max_steps):
        a = expert_policy_masked(env, obs)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return float(total)

def run_week_ppo(seed: int, model, max_steps=WEEK_STEPS) -> float:
    env = WorkshopEnv()
    obs, _ = env.reset(seed=seed)
    total = 0.0

    for _ in range(max_steps):
        mask = env.get_action_mask()
        a, _ = model.predict(obs, deterministic=True, action_masks=mask)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return float(total)

# Petit sanity check sur 2 seeds
for s in test_seeds[:2]:
    print(s, "expert", run_week_expert(s), "| ppo", run_week_ppo(s, ppo_v7))


50000 expert 14498.339999998527 | ppo 16298.699999997283
50001 expert 14135.87999999846 | ppo 15310.799999997495


In [13]:
# Cellule 6 — Lancer la comparaison sur 104 semaines 
# + sauvegarder CSV

rows = []
for i, seed in enumerate(test_seeds, start=1):
    r_exp = run_week_expert(seed)
    r_ppo = run_week_ppo(seed, ppo_v7)

    rows.append({
        "week_id": i,
        "seed": seed,
        "reward_expert_v4": r_exp,
        "reward_ppo_v7": r_ppo,
        "diff_ppo_minus_expert": (r_ppo - r_exp)
    })

    if i % 10 == 0:
        print(f"{i}/{N_WEEKS} done...")

df = pd.DataFrame(rows)
display(df.head())

out_csv = "cmp_expertv4_vs_ppo_v7_104weeks.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


10/104 done...
20/104 done...
30/104 done...
40/104 done...
50/104 done...
60/104 done...
70/104 done...
80/104 done...
90/104 done...
100/104 done...


,week_id,seed,reward_expert_v4,reward_ppo_v7,diff_ppo_minus_expert
0,1,50000,12833.80,15761.48,2927.68
1,2,50001,13077.22,15123.32,2046.10
2,3,50002,13094.10,15965.80,2871.70
3,4,50003,13790.94,16358.52,2567.58
4,5,50004,13382.12,15755.02,2372.90


Saved: cmp_expertv4_vs_ppo_v7_104weeks.csv


In [16]:
# Cellule 7 — Résumé chiffré + taux de victoire

summary = pd.DataFrame({
    "metric": [
        "mean_reward",
        "std_reward",
        "min_reward",
        "p25_reward",
        "median_reward",
        "p75_reward",
        "max_reward",
    ],
    "expert_v4": [
        df["reward_expert_v4"].mean(),
        df["reward_expert_v4"].std(),
        df["reward_expert_v4"].min(),
        df["reward_expert_v4"].quantile(0.25),
        df["reward_expert_v4"].median(),
        df["reward_expert_v4"].quantile(0.75),
        df["reward_expert_v4"].max(),
    ],
    "ppo_v7": [
        df["reward_ppo_v7"].mean(),
        df["reward_ppo_v7"].std(),
        df["reward_ppo_v7"].min(),
        df["reward_ppo_v7"].quantile(0.25),
        df["reward_ppo_v7"].median(),
        df["reward_ppo_v7"].quantile(0.75),
        df["reward_ppo_v7"].max(),
    ],
})

win_rate = (df["diff_ppo_minus_expert"] > 0).mean()

print("Win-rate PPO > Expert :", f"{100*win_rate:.1f}%")
display(summary)


Win-rate PPO > Expert : 100.0%


,metric,expert_v4,ppo_v7
0,mean_reward,13310.849038,15768.534231
1,std_reward,553.106242,387.050096
2,min_reward,11831.620000,14811.860000
3,p25_reward,12957.830000,15481.710000
4,median_reward,13282.560000,15767.830000
5,p75_reward,13611.115000,16053.225000
6,max_reward,15058.380000,16872.320000
